In [13]:
from typing import TypedDict, Annotated
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq

load_dotenv()

True

In [14]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7
)

In [15]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [16]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [17]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [18]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)



In [19]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'burger'}, config=config1)

{'topic': 'burger',
 'joke': 'Why did the burger go to therapy? \n\nBecause it was feeling a little "crumby" and had a lot of "beef" to work through.',
 'explanation': 'This joke is a play on words. It starts with the question "Why did the burger go to therapy?" which sets up an expectation that the answer will be a pun related to the burger\'s situation or emotions. Instead, it uses a common phrase "feeling a little crumby" which is a common idiomatic expression meaning feeling unwell or down. However, in this case, it\'s also a clever pun as burgers often have crumbly toppings like buns and it also references the fact that the burger itself might be crumbling or falling apart.\n\nThe second part of the joke "had a lot of beef to work through" is another clever play on words. A common idiomatic expression is "having a lot of beef with someone" meaning to have a disagreement or a problem with them. However, in this case, it\'s a pun on the fact that a burger is typically made of beef, 

In [20]:

workflow.get_state(config1)

StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to therapy? \n\nBecause it was feeling a little "crumby" and had a lot of "beef" to work through.', 'explanation': 'This joke is a play on words. It starts with the question "Why did the burger go to therapy?" which sets up an expectation that the answer will be a pun related to the burger\'s situation or emotions. Instead, it uses a common phrase "feeling a little crumby" which is a common idiomatic expression meaning feeling unwell or down. However, in this case, it\'s also a clever pun as burgers often have crumbly toppings like buns and it also references the fact that the burger itself might be crumbling or falling apart.\n\nThe second part of the joke "had a lot of beef to work through" is another clever play on words. A common idiomatic expression is "having a lot of beef with someone" meaning to have a disagreement or a problem with them. However, in this case, it\'s a pun on the fact that a burger is typic

In [21]:


list(workflow.get_state_history(config1))



[StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to therapy? \n\nBecause it was feeling a little "crumby" and had a lot of "beef" to work through.', 'explanation': 'This joke is a play on words. It starts with the question "Why did the burger go to therapy?" which sets up an expectation that the answer will be a pun related to the burger\'s situation or emotions. Instead, it uses a common phrase "feeling a little crumby" which is a common idiomatic expression meaning feeling unwell or down. However, in this case, it\'s also a clever pun as burgers often have crumbly toppings like buns and it also references the fact that the burger itself might be crumbling or falling apart.\n\nThe second part of the joke "had a lot of beef to work through" is another clever play on words. A common idiomatic expression is "having a lot of beef with someone" meaning to have a disagreement or a problem with them. However, in this case, it\'s a pun on the fact that a burger is typi

Fault Tolerance

In [10]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [11]:

# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str


In [ ]:
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [ ]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

In [ ]:

print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


Time Travel

In [23]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f170682-4d06-64b5-8000-a9afb3b92068"}})

StateSnapshot(values={'topic': 'burger'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f170682-4d06-64b5-8000-a9afb3b92068'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-25T07:33:22.174251+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f170682-4c54-63da-bfff-48f6e772c495'}}, tasks=(PregelTask(id='73d658ee-d01d-c1e7-4e00-5138df96f1a2', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the burger go to therapy? \n\nBecause it was feeling a little "crumby" and had a lot of "beef" to work through.'}),), interrupts=())

In [29]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f170682-4d06-64b5-8000-a9afb3b92068"}})

{'topic': 'burger',
 'joke': 'Why did the burger go to therapy? \n\nBecause it was feeling a little "well done" under the pressure.',
 'explanation': 'The joke relies on a play on words. "Well done" is a term often used to describe a burger that has been cooked to a specific level of doneness, particularly browned and crispy. \n\nHowever, in a therapeutic context, "feeling well done" would mean that someone feels good or successful after going through a challenging experience. \n\nThe punchline is humorous because it takes the common understanding of "well done" in cooking and applies it to the burger\'s emotional state, creating a clever and unexpected connection between the two. The wordplay creates a lighthearted and amusing effect.'}

In [31]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to therapy? \n\nBecause it was feeling a little "well done" under the pressure.', 'explanation': 'The joke relies on a play on words. "Well done" is a term often used to describe a burger that has been cooked to a specific level of doneness, particularly browned and crispy. \n\nHowever, in a therapeutic context, "feeling well done" would mean that someone feels good or successful after going through a challenging experience. \n\nThe punchline is humorous because it takes the common understanding of "well done" in cooking and applies it to the burger\'s emotional state, creating a clever and unexpected connection between the two. The wordplay creates a lighthearted and amusing effect.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f170690-f658-61a4-8003-e865a41cd96d'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-06-25T07:39:55.738324+00:00

updating state

In [30]:


workflow.update_state({"configurable": {"thread_id": "1f170688-288e-691f-8003-da311a167b0b", "checkpoint_ns": ""}}, {'topic':'samosa'})



{'configurable': {'thread_id': '1f170688-288e-691f-8003-da311a167b0b',
  'checkpoint_ns': '',
  'checkpoint_id': '1f170692-7336-6516-8000-be2c0a546ba5'}}